In [ ]:
# # 📘 Lesson 07 — Data Leakage
# - Kaggle Intermediate Machine Learning — Study Companion (Moacir)  
# - Hybrid Learning Notebook — Study + Reference + Hands-on

## 📚 Curso Intermediate Machine Learning — Lesson 07: Data Leakage

Nesta lesson, vamos estudar um dos problemas mais perigosos em Machine Learning:

> **Data Leakage** — quando informações que não estarão disponíveis em produção “vazam” para o treinamento ou validação, fazendo o modelo parecer melhor do que realmente é.

Esta lesson acompanha diretamente a lição **Data Leakage** do Kaggle Intermediate ML.

---
### 🎯 Objetivos da Lesson

- Entender o que é **data leakage** e por que ele é tão perigoso.
- Diferenciar os dois tipos principais:
  - **Target leakage**
  - **Train-test contamination**
- Ver um exemplo prático de target leakage em um problema de classificação.
- Aprender uma forma sistemática de:
  - suspeitar de leakage
  - investigar variáveis suspeitas
  - remover preditores com vazamento
- Conectar leakage com:
  - **validação** (Cross‑Validation)
  - **pipelines** (separação correta de treino/validação)

---
### 🟦 Glossário Técnico

- **Data Leakage (leakage)** — situação em que o modelo tem acesso, durante o treino, a informações que não estarão disponíveis no momento de previsão em produção.
- **Target Leakage** — tipo de leakage em que um preditor contém, direta ou indiretamente, informação sobre o alvo *após* o evento ter acontecido.
- **Train-Test Contamination** — tipo de leakage em que dados de validação/teste influenciam o pré‑processamento ou o treinamento.
- **Production (produção)** — ambiente real onde o modelo é usado para tomar decisões com dados novos.
- **Cross‑Validation** — técnica de avaliação que usa múltiplas divisões treino/validação.
- **RandomForestClassifier** — modelo de classificação baseado em múltiplas árvores de decisão.
- **Accuracy** — proporção de previsões corretas em um problema de classificação.

---
### 🟩 Mini‑Referência (API Style)

- **Modelos e Pipelines**
  - `from sklearn.ensemble import RandomForestClassifier`
  - `from sklearn.pipeline import make_pipeline`

- **Validação**
  - `from sklearn.model_selection import cross_val_score`
  - `cross_val_score(model, X, y, cv=5, scoring='accuracy')`

- **Operações com pandas**
  - `X.drop(columns, axis=1)` — remove colunas.
  - `X[col][mask]` — seleciona valores de uma coluna com base em uma condição.
  - `(series == 0).mean()` — fração de valores iguais a zero.

- **Conceito central da lesson**
  - Investigar variáveis suspeitas de leakage com comparações simples:
    - frações
    - distribuições
    - relação com o target

---
### 🟧 Introdução Conceitual — O que é Data Leakage?

**Data leakage** acontece quando o modelo aprende, durante o treinamento, algo que ele **não poderia saber** no momento em que será usado em produção.

Isso gera um efeito perigoso:

- o modelo parece excelente em:
  - treino
  - validação
  - até mesmo em cross‑validation
- mas falha de forma grave quando é colocado em produção.

Em outras palavras:

> Leakage faz o modelo “trapacear” durante o treino, usando pistas que não existirão no mundo real.

A lesson do Kaggle foca em dois tipos principais:

1. **Target Leakage**  
   Quando um preditor é atualizado *depois* que o target é definido.

2. **Train-Test Contamination**  
   Quando dados de validação/teste influenciam:
   - imputação
   - normalização
   - feature engineering
   - qualquer etapa de pré‑processamento.

Esta lesson é complementar às anteriores:

- **Pipelines** ajudam a evitar train-test contamination.
- **Cross‑Validation** continua válido, mas pode ser enganado se houver leakage.

---
### 🟨 Parte 1 — Target Leakage (Exemplo Conceitual)

A lesson começa com um exemplo de saúde:

- Queremos prever quem vai ter **pneumonia**.
- Temos, entre outras colunas:
  - `got_pneumonia` (target)
  - `age`, `weight`, `male`
  - `took_antibiotic_medicine`

As pessoas tomam antibiótico **depois** de terem pneumonia.

Se usarmos `took_antibiotic_medicine` como feature:

- o modelo aprende:
  - “se tomou antibiótico → provavelmente teve pneumonia”
- isso funciona bem em:
  - treino
  - validação (que vem da mesma fonte de dados)
- mas falha em produção, porque:
  - no momento da previsão, o paciente **ainda não tomou** antibiótico.

**Regra prática importante:**

> Qualquer variável criada ou atualizada *depois* que o target é definido é candidata forte a target leakage.

---
### 🟪 Parte 2 — Train-Test Contamination (Conceito)

O segundo tipo de leakage é mais sutil:

> **Train-test contamination** acontece quando dados de validação/teste influenciam o pré‑processamento ou o treinamento.

Exemplo típico:

- Você faz imputação de missing values **antes** de dividir em treino/validação:
  - o imputador “vê” todo o dataset
  - inclusive as linhas que deveriam ser usadas apenas para validação
- Depois, você faz `train_test_split` e treina o modelo.

O problema:

- o modelo foi treinado com um pré‑processamento que já incorporou informação do conjunto de validação.
- a validação deixa de ser “dados nunca vistos”.

A lesson reforça:

- **Pipelines** ajudam a evitar esse problema:
  - o `fit` é feito apenas nos dados de treino.
  - o `transform` é aplicado separadamente em validação/teste.
- Em **Cross‑Validation**, isso é ainda mais crítico:
  - todo pré‑processamento deve estar dentro do pipeline.

---
### 🟥 Parte 3 — Exemplo Prático de Target Leakage (Cartões de Crédito)

Agora vamos seguir o exemplo prático da lesson:

- Dataset: aplicações de **cartão de crédito**.
- Objetivo: prever se uma aplicação será **aprovada** ou não.
- Target: `card` (1 = aprovado, 0 = não aprovado).
- Features (entre outras):
  - `reports`, `age`, `income`, `share`, `expenditure`,
  - `owner`, `selfemp`, `dependents`, `months`,
  - `majorcards`, `active`.

A lesson mostra que um modelo inicial atinge **~98% de acurácia** em cross‑validation — o que é suspeito.

Vamos reproduzir a lógica geral do exemplo.

---
### 🟨 Importar Bibliotecas + Carregar Dados

Aqui seguimos o mesmo padrão de organização dos notebooks anteriores:

- usamos `config.py` para obter o caminho base dos dados (`data/raw/`);
- carregamos o dataset de cartão de crédito em um `DataFrame X`;
- carregamos o target `y` (coluna `card`).

#### 1

In [1]:
import sys
from pathlib import Path

# Caminho absoluto do notebook
notebook_path = Path().resolve()

# Sobe diretórios até encontrar config.py
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

In [2]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from config import get_data_path, get_output_path

DATA_PATH = get_data_path()
OUTPUT_PATH = get_output_path()

> ⚠️ Observação  
O nome exato do arquivo pode variar no seu ambiente.  
Ajuste a string abaixo (`credit-card-data.csv`) conforme o nome real do dataset de cartões de crédito que você estiver usando.

#### 2 Dataset sintético da Lesson 07 — pronto para uso

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Caminho onde o dataset será salvo
output_path = Path("../data/raw/credit-card-data.csv")

# Número de linhas (igual ao Kaggle)
n = 1319
rng = np.random.default_rng(42)

# -----------------------------
# 1) Gerar target (card)
# -----------------------------
# Aprovação ~30% (valor aproximado do dataset real)
card = rng.choice([0, 1], size=n, p=[0.7, 0.3])

# -----------------------------
# 2) Variáveis numéricas
# -----------------------------
reports = rng.poisson(0.5, size=n)
age = rng.normal(35, 10, size=n).clip(18, 80)
income = rng.normal(5, 2, size=n).clip(0.5, 20)

dependents = rng.integers(1, 6, size=n)
months = rng.integers(1, 120, size=n)

# -----------------------------
# 3) Variáveis booleanas
# -----------------------------
owner = rng.choice([0, 1], size=n, p=[0.4, 0.6])
selfemp = rng.choice([0, 1], size=n, p=[0.9, 0.1])

# -----------------------------
# 4) Variáveis com leakage
# -----------------------------
# expenditure = gasto mensal no cartão
# No dataset real:
# - quem NÃO recebeu cartão → gasto = 0
# - quem recebeu → gasto > 0 (quase sempre)
expenditure = np.where(
    card == 0,
    0,
    rng.normal(150, 80, size=n).clip(0, None)
)

# share = expenditure / income anual
share = expenditure / (income * 10000 + 1)

# majorcards e active também correlacionam com aprovação
majorcards = np.where(card == 1,
                      rng.integers(1, 4, size=n),
                      rng.integers(0, 2, size=n))

active = np.where(card == 1,
                  rng.integers(5, 15, size=n),
                  rng.integers(0, 5, size=n))

# -----------------------------
# 5) Montar DataFrame final
# -----------------------------
df = pd.DataFrame({
    "reports": reports,
    "age": age.round(2),
    "income": income.round(2),
    "share": share.round(6),
    "expenditure": expenditure.round(2),
    "owner": owner.astype(int),
    "selfemp": selfemp.astype(int),
    "dependents": dependents,
    "months": months,
    "majorcards": majorcards,
    "active": active,
    "card": card
})

# -----------------------------
# 6) Salvar dataset
# -----------------------------
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print("Dataset sintético salvo em:", output_path)
df.head()

3

In [5]:
# Carregar dataset de aplicações de cartão de crédito
credit_path = DATA_PATH + "credit-card-data.csv"  # ajuste o nome se necessário
data = pd.read_csv(credit_path)

# Separar features (X) e target (y)
y = data["card"]          # 1 = aprovado, 0 = não aprovado
X = data.drop(["card"], axis=1)

print("Número de linhas no dataset:", len(X))
X.head()

Número de linhas no dataset: 1319


,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,1,42.04,5.58,0.003617,201.84,1,0,3,4,3,12
1,0,24.75,8.59,0.000000,0.00,0,1,5,112,0,3
2,0,45.17,2.00,0.000401,8.03,0,0,5,110,2,13
3,1,39.66,4.98,0.000000,0.00,0,0,3,21,1,1
4,1,21.83,5.73,0.000000,0.00,0,0,4,38,1,3


---
### 🟫 Construir Pipeline + Cross‑Validation (Modelo Inicial)

A lesson usa:

- `RandomForestClassifier` como modelo base;
- `make_pipeline` para criar um pipeline simples (mesmo sem pré‑processamento);
- `cross_val_score` com:
  - `cv=5`
  - `scoring='accuracy'`.

A ideia é:

- medir a acurácia média em 5 folds;
- observar que o valor é **muito alto** (por volta de 0.98);
- suspeitar de leakage.

In [6]:
# Pipeline simples com RandomForestClassifier
my_pipeline = make_pipeline(
    RandomForestClassifier(n_estimators=100, random_state=0)
)

# Cross‑Validation com 5 folds
cv_scores = cross_val_score(
    my_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation accuracy: %f" % cv_scores.mean())

Cross-validation accuracy: 1.000000


---
### ⭐ Interpretação da Acurácia Muito Alta

A lesson destaca:

- Modelos com **98% de acurácia** em problemas reais são raros.
- Quando isso acontece, é importante:
  - desconfiar dos dados;
  - investigar possíveis vazamentos.

Em vez de assumir que o modelo é “perfeito”, o Kaggle sugere:

> “Se a métrica estiver boa demais para ser verdade, investigue leakage.”

---
### 🟦 Entendendo as Features do Dataset

A lesson descreve as colunas principais:

- `card` — 1 se a aplicação foi aceita, 0 caso contrário.
- `reports` — número de relatórios de crédito negativos.
- `age` — idade (anos + fração de ano).
- `income` — renda anual (dividida por 10.000).
- `share` — razão entre gasto mensal no cartão e renda anual.
- `expenditure` — gasto médio mensal no cartão.
- `owner` — 1 se é dono do imóvel, 0 se aluga.
- `selfemp` — 1 se autônomo, 0 caso contrário.
- `dependents` — 1 + número de dependentes.
- `months` — meses no endereço atual.
- `majorcards` — número de cartões de crédito principais.
- `active` — número de contas de crédito ativas.

Algumas variáveis chamam atenção:

- `expenditure` e `share` parecem depender do uso do cartão.
- `majorcards` e `active` podem refletir comportamento de crédito após a aprovação.

---
### 🟩 Investigando uma Variável Suspeita: `expenditure`

A lesson sugere uma análise simples:

- comparar a distribuição de `expenditure` entre:
  - quem recebeu o cartão (`card == 1`)
  - quem não recebeu (`card == 0`)

Em particular:

- qual a fração de pessoas com `expenditure == 0` em cada grupo?

In [7]:
# Separar gastos de quem recebeu e de quem não recebeu o cartão
expenditures_cardholders = X.loc[y == 1, "expenditure"]
expenditures_noncardholders = X.loc[y == 0, "expenditure"]

print(
    "Fraction of those who did not receive a card and had no expenditures: %.2f"
    % ((expenditures_noncardholders == 0).mean())
)
print(
    "Fraction of those who received a card and had no expenditures: %.2f"
    % ((expenditures_cardholders == 0).mean())
)

Fraction of those who did not receive a card and had no expenditures: 1.00
Fraction of those who received a card and had no expenditures: 0.02


---
### ⭐ Interpretação da Análise de `expenditure`

A lesson mostra um padrão extremo:

- **100%** das pessoas que **não receberam** o cartão têm `expenditure == 0`.
- Apenas **~2%** das pessoas que **receberam** o cartão têm `expenditure == 0`.

Isso sugere fortemente que:

- `expenditure` representa gastos **no próprio cartão aplicado**.
- Logo, essa informação só existe **depois** da aprovação.

Portanto:

> `expenditure` é um caso clássico de **target leakage**.

---
### 🟨 Removendo Preditores com Vazamento

A lesson segue com uma decisão conservadora:

- remover não só `expenditure`, mas também:
  - `share` (derivada de `expenditure`)
  - `active`
  - `majorcards`

A lógica:

- se não temos certeza de que a variável é “segura” do ponto de vista temporal,
- é melhor removê‑la do que correr o risco de leakage.

In [8]:
# Lista de potenciais preditores com leakage
potential_leaks = ["expenditure", "share", "active", "majorcards"]

# Criar nova matriz de features sem essas colunas
X2 = X.drop(potential_leaks, axis=1)

X2.head()

,reports,age,income,owner,selfemp,dependents,months
0,1,42.04,5.58,1,0,3,4
1,0,24.75,8.59,0,1,5,112
2,0,45.17,2.00,0,0,5,110
3,1,39.66,4.98,0,0,3,21
4,1,21.83,5.73,0,0,4,38


---
### 🟫 Reavaliando o Modelo sem Leakage

Agora repetimos a mesma avaliação:

- mesmo pipeline (`RandomForestClassifier`);
- mesma estratégia de validação (`cross_val_score`, `cv=5`, `accuracy`);
- mas usando `X2` (sem as colunas suspeitas).

In [9]:
cv_scores_no_leak = cross_val_score(
    my_pipeline,
    X2,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-val accuracy (no leakage): %f" % cv_scores_no_leak.mean())

Cross-val accuracy (no leakage): 0.657305


---
### ⭐ Comparando Resultados

- Com leakage:
  - acurácia ≈ 0.98
- Sem leakage:
  - acurácia ≈ 0.83 (valor típico mostrado na lesson)

A conclusão da lesson:

- o modelo com leakage **parece melhor**, mas é ilusório;
- o modelo sem leakage:
  - tem desempenho menor na métrica,
  - mas é muito mais confiável em produção.

> Em Machine Learning aplicado, **confiabilidade em produção** é mais importante do que uma métrica artificialmente alta em validação.

---
### 🟦 Parte 4 — Exercícios Conceituais (Resumo)

A lesson termina com uma série de cenários conceituais para treinar a intuição sobre leakage:

1. **Shoelaces (cadarços)**  
   - Prever quantos cadarços serão necessários.  
   - Feature suspeita: quantidade de couro **usado** no mês (pode depender da produção já realizada).

2. **Shoelaces 2 — couro encomendado**  
   - Usar a quantidade de couro **encomendado** pode ser aceitável, dependendo de:
     - se o pedido é feito antes da produção;
     - se essa informação está disponível no momento da previsão.

3. **Criptomoedas**  
   - Modelo com erro médio < 1 em um ativo que varia mais de 100 unidades.  
   - Suspeita de leakage ou problema de avaliação (por exemplo, prever o preço de amanhã usando informação do próprio amanhã).

4. **Infecções pós‑cirurgia**  
   - Usar a taxa média de infecção por cirurgião como feature.  
   - Pode haver:
     - target leakage (se a taxa for calculada usando o próprio paciente);
     - train-test contamination (se a taxa for calculada usando todo o dataset, incluindo validação).

5. **Preços de casas**  
   - Feature suspeita: preço médio de casas no mesmo bairro, calculado com dados que incluem o próprio imóvel alvo.

Esses exercícios reforçam a ideia central:

> Leakage é, em grande parte, um problema de **linha do tempo** e **definição correta do que está disponível no momento da previsão**.

---
### 🟩 Observações Pedagógicas do Copilot

- A lesson assume que você já entende:
  - **validação** (train/validation split, cross‑validation);
  - **pipelines** (separação de pré‑processamento e modelo).

- O Kaggle não enfatiza tanto, mas é importante lembrar:
  - leakage não é detectado automaticamente por nenhuma função do scikit‑learn;
  - métricas altas **não garantem** que o modelo é saudável;
  - entender o **domínio do problema** é essencial para identificar variáveis suspeitas.

- Conexão com lessons anteriores:
  - **Pipelines** ajudam a evitar train-test contamination:
    - o `fit` de imputadores/encoders é feito apenas em treino.
  - **Cross‑Validation** continua sendo a melhor forma de estimar desempenho,
    mas **não protege contra leakage de target**.

- Intuição prática:
  - sempre pergunte:
    - “Essa informação estaria disponível no momento em que o modelo será usado?”
    - “Essa feature depende de algo que acontece depois do evento alvo?”

---
### 🧠 Conclusão da Lesson 07 — Data Leakage

Nesta lesson, você viu que:

- **Data leakage** é um dos problemas mais perigosos em Machine Learning aplicado.
- Existem dois tipos principais:
  - **Target leakage** — preditores que usam informação do futuro (após o target).
  - **Train-test contamination** — pré‑processamento influenciado por dados de validação/teste.
- Métricas muito altas podem ser um sinal de alerta, não de sucesso.
- Um modelo com métrica mais modesta, mas sem leakage, é **muito mais confiável** em produção.
- Pipelines e uma boa organização do fluxo de dados ajudam a reduzir riscos de leakage, mas não substituem:
  - entendimento do domínio;
  - análise crítica das features.

Próximo passo:

- Revisar os exercícios conceituais da lesson no Kaggle.
- Praticar identificar leakage em diferentes cenários.
- Levar essa mentalidade para qualquer projeto real de Machine Learning que você construir.